# Truth Archive EDA (Remote Parquet)

This notebook performs core-profile EDA by fetching data directly from Trump Truth Social Parquet archive: `https://ix.cnn.io/data/truth-social/truth_archive.parquet`.

## Setup

In [ ]:
import io

import pandas as pd
import plotly.express as px
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Load Data

In [ ]:
ARCHIVE_URL = "https://ix.cnn.io/data/truth-social/truth_archive.parquet"
HTTP_HEADERS = {"User-Agent": "ttsfeed/0.1 (Truth Social archive tracker)"}
TIMEOUT_SECONDS = 120

resp = requests.get(ARCHIVE_URL, timeout=TIMEOUT_SECONDS, headers=HTTP_HEADERS)
resp.raise_for_status()

raw_bytes = resp.content
df_raw = pd.read_parquet(io.BytesIO(raw_bytes))


print(f"Fetched {len(raw_bytes) / 1_000_000:.2f} MB from {ARCHIVE_URL}")

Fetched 5.68 MB from https://ix.cnn.io/data/truth-social/truth_archive.parquet
Rows: 31,625 | Columns: 8


## Data Cleaning

In [37]:
df = df_raw.replace("", pd.NA)

In [66]:
df['created_at'] = pd.to_datetime(df['created_at'])

## Schema & Missing Values

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31625 entries, 0 to 31624
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                31625 non-null  object             
 1   created_at        31625 non-null  datetime64[ns, UTC]
 2   content           25983 non-null  object             
 3   url               31625 non-null  object             
 4   media             10903 non-null  object             
 5   replies_count     31625 non-null  int64              
 6   reblogs_count     31625 non-null  int64              
 7   favourites_count  31625 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(3), object(4)
memory usage: 1.9+ MB


In [68]:
missing_df = (
    df.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_pct"})
    .reset_index(names="column")
)
missing_df["missing_pct"] = missing_df["missing_pct"].round(6)
missing_df = missing_df.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)
missing_df

,column,missing_count,missing_pct
0,media,20722.0,0.655241
1,content,5642.0,0.178403
2,created_at,0.0,0.000000
3,favourites_count,0.0,0.000000
4,id,0.0,0.000000
5,reblogs_count,0.0,0.000000
6,replies_count,0.0,0.000000
7,url,0.0,0.000000


In [69]:
(df['media'].isna() & df['content'].isna()).sum()

2

In [70]:
df[df['media'].isna() & df['content'].isna()]

,id,created_at,content,url,media,replies_count,reblogs_count,favourites_count
15384,112044239738550427,2024-03-05 17:23:39.830000+00:00,<NA>,https://truthsocial.com/@realDonaldTrump/11204...,<NA>,63,480,3098
15385,112044239731983483,2024-03-05 17:23:39.730000+00:00,<NA>,https://truthsocial.com/@realDonaldTrump/11204...,<NA>,157,888,6508


## Summary Statistics

In [71]:
df.describe()

,replies_count,reblogs_count,favourites_count
count,31625.000000,31625.000000,31625.000000
mean,770.361202,3405.426213,13214.092142
std,1323.476291,3228.775383,12316.103099
min,0.000000,0.000000,0.000000
25%,120.000000,1464.000000,5629.000000
50%,396.000000,2870.000000,11199.000000
75%,961.000000,4629.000000,18054.000000
max,60886.000000,123196.000000,410837.000000


In [ ]:
df["created_at"].describe(datetime_is_numeric=True)

count                                  31625
mean     2024-03-18 08:16:03.358105088+00:00
min         2022-02-14 15:54:32.528000+00:00
25%      2023-06-30 18:01:09.116999936+00:00
50%      2024-02-19 18:41:21.723000064+00:00
75%      2024-11-04 19:54:36.153999872+00:00
max         2026-02-21 04:32:29.567000+00:00
Name: created_at, dtype: object

## Temporal Analysis

In [104]:
posts_by_date = df.assign(created_date=df["created_at"].dt.date)['created_date'].value_counts().sort_index()

In [107]:
px.line(
    posts_by_date,
    labels={"index": "Date", "value": "Number of Posts"},
    title="Posts by Date",
).update_layout(showlegend=False)

In [96]:
posts_per_hour_utc = df.assign(created_hour=df["created_at"].dt.hour)['created_hour'].value_counts().sort_index()
posts_per_hour_ny = (
    df.assign(
        created_hour_ny=(
            df["created_at"]
            .dt.tz_convert("America/New_York")
            .dt.hour
        )
    )["created_hour_ny"]
    .value_counts()
    .sort_index()
)

In [ ]:
px.bar(
    posts_per_hour_utc,
    labels={"index": "Hour of Day", "value": "Number of Posts"},
    title="Posts by Hour (UTC)",
)

In [ ]:
px.bar(
    posts_per_hour_ny,
    labels={"index": "Hour of Day", "value": "Number of Posts"},
    title="Posts by Hour (NY)",
)